## Bab 1: Persiapan Environment & Instalasi Library
Menginstal pustaka LangChain, ChromaDB (Vector Store), pemroses PDF, dan dependensi model.

In [1]:
# Instalasi library utama untuk RAG
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q pypdf chromadb sentence-transformers
!pip install -q transformers accelerate bitsandbytes

print("✅ Semua library RAG berhasil diinstal!")

✅ Semua library RAG berhasil diinstal!


## Bab 2: Memuat & Memproses Dokumen PDF Undang-Undang
Sesuai kriteria Dicoding, kita wajib menggunakan 4 dokumen PDF Undang-Undang sebagai basis pengetahuan (konteks) agar AI tidak berhalusinasi.

In [2]:
import os
import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. MENCARI SEMUA PDF DI SELURUH FOLDER INPUT KAGGLE
# (Menggunakan glob agar file terdeteksi walau nama foldernya tersembunyi)
pdf_files = glob.glob('/kaggle/input/**/*.pdf', recursive=True)

documents = []

print(f"🔍 Ditemukan {len(pdf_files)} file PDF di sistem Kaggle. Sedang memproses...\n")

# 2. Membaca isi teks dari setiap PDF yang ditemukan
for file_path in pdf_files:
    loader = PyPDFLoader(file_path)
    documents.extend(loader.load())
    print(f"✅ Berhasil memuat: {os.path.basename(file_path)}")

# 3. Memotong teks menjadi chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(documents)
print(f"\n🚀 SUCCESS! Total potongan teks yang siap dimasukkan ke database: {len(chunks)} chunks")

/tmp/ipykernel_614/3309970964.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


🔍 Ditemukan 4 file PDF di sistem Kaggle. Sedang memproses...

✅ Berhasil memuat: PP Nomor 5 Tahun 2021.pdf
✅ Berhasil memuat: UU Nomor 6 Tahun 2023.pdf
✅ Berhasil memuat: PP Nomor 51 Tahun 2023.pdf
✅ Berhasil memuat: PP Nomor 35 Tahun 2021.pdf

🚀 SUCCESS! Total potongan teks yang siap dimasukkan ke database: 3242 chunks


## Bab 3: Vector Database & Embeddings
Mengubah potongan dokumen (chunks) menjadi representasi vektor menggunakan model open-source dan menyimpannya di ChromaDB agar mudah dicari oleh AI saat ada pertanyaan.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Tahap 1: Sedang mengunduh model embedding...")
# 1. Menggunakan model embedding open-source yang ringan dan cepat
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Tahap 2: Sedang membangun Vector Database (ini butuh waktu 1-2 menit)...")
# 2. Memasukkan seluruh chunks ke dalam database Chroma
vectorstore = Chroma.from_documents(
    documents = chunks, 
    embedding = embedding_model,
    persist_directory = "./chroma_db_legal"
)

# 3. Mengatur sistem pencari (Retriever) untuk mengambil 3 dokumen paling relevan
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("\n✅ ALHAMDULILLAH! Vector Database berhasil dibuat dan Retriever siap digunakan!")

Tahap 1: Sedang mengunduh model embedding...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tahap 2: Sedang membangun Vector Database (ini butuh waktu 1-2 menit)...

✅ ALHAMDULILLAH! Vector Database berhasil dibuat dan Retriever siap digunakan!


## Bab 4: Memuat Model Llama-3 Hasil Fine-Tuning
Memanggil model yang sudah di-fine-tune sebelumnya dari Hugging Face Hub untuk digunakan sebagai generator jawaban.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from kaggle_secrets import UserSecretsClient

# 1. Mengambil token Hugging Face secara aman
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# 2. Tentukan ID model hasil fine-tuning Anda
model_id = "vikriahaikal/llama3-legal-bot"

print("Sedang memuat model Llama-3 (ini butuh waktu sekitar 2-3 menit)...")

# 3. Konfigurasi 4-bit (Cara baru sesuai update Transformers)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 4. Memuat Tokenizer dan Model dengan konfigurasi baru
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",
    quantization_config=bnb_config # Memasukkan setingan 4-bit di sini
)

# 5. Membuat Pipeline Transformers
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.15
)

# 6. Integrasi ke LangChain
llm = HuggingFacePipeline(pipeline=pipe)

print("\n✅ Model Llama-3 berhasil dimuat!")

Sedang memuat model Llama-3 (ini butuh waktu sekitar 2-3 menit)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'temperature', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



✅ Model Llama-3 berhasil dimuat!


## Bab 5: Membangun Chain RAG & Uji Coba Pertanyaan
Menggabungkan database dokumen dengan model AI. AI akan mencari konteks di dokumen dulu sebelum menjawab.

In [5]:
# 0. Instalasi modul klasik LangChain yang dipisah oleh developer
!pip install -q langchain-classic

import sys
from IPython.display import display, Markdown
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# 1. Template Prompt yang SUDAH DISESUAIKAN dengan Token Spesial Llama-3
template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Anda adalah asisten hukum tim legal yang kompeten. Tugas Anda adalah menjawab pertanyaan user secara informatif, singkat, dan HANYA berdasarkan potongan konteks dokumen yang disediakan. 
Jika Anda tidak tahu jawabannya atau informasi tidak ada di dalam konteks, katakan saja Anda tidak tahu. Jangan mencoba mengarang jawaban atau mengambil sumber dari luar konteks dokumen.<|eot_id|><|start_header_id|>user<|end_header_id|>
Gunakan potongan konteks berikut untuk menjawab pertanyaan di akhir. Jaga jawaban maksimal tiga kalimat dan sesingkat mungkin.

Konteks: {context}

Pertanyaan: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Jawaban Informatif:"""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

# 2. Membuat RAG Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

# 3. INTERACTIVE INTERFACE (Memenuhi Syarat Wajib Dicoding)
print("=======================================================")
print("🤖 Bot Hukum Legal Team (Llama-3 Secured Version) Ready!")
print("Ketik 'keluar' untuk menghentikan percakapan interaktif.")
print("=======================================================\n")

while True:
    question = input("Masukkan Pertanyaan Hukum Anda: ")
    
    if question.lower() == 'keluar':
        print("\nSesi interaktif dihentikan. Terima kasih!")
        break
        
    if not question.strip():
        continue
        
    print("\n🔍 Sedang mencari dasar hukum di tumpukan dokumen...")
    
    # Eksekusi RAG
    result = qa_chain.invoke({"query": question})
    
    # Membersihkan teks sisa template jika ada yang bocor keluar
    clean_result = result['result'].split("Jawaban Informatif:")[-1].strip()
    
    # Menampilkan output dalam format Markdown yang rapi
    display(Markdown(f"### 🤖 AI Response:\n{clean_result}"))
    
    print("\n📚 Sumber Dokumen:")
    for doc in result["source_documents"]:
        file_name = doc.metadata['source'].split('/')[-1]
        print(f"- {file_name} (Halaman {doc.metadata['page']})")
    print("\n" + "-"*60 + "\n")

🤖 Bot Hukum Legal Team (Llama-3 Secured Version) Ready!
Ketik 'keluar' untuk menghentikan percakapan interaktif.



Masukkan Pertanyaan Hukum Anda:  Apa saja jenis Perizinan Berusaha Berbasis Risiko?


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Sedang mencari dasar hukum di tumpukan dokumen...


### 🤖 AI Response:
Menurut konteks yang diberikan, ada empat jenis Perizinan Berusaha Berbasis Risiko:

1. NIB (Nomor Induk Berusaha): Jenis ini digunakan untuk aktivitas risiko rendah.
2. Sertifikat Standar: Digunakan untuk aktivitas risiko menengah rendah.
3. Izin: Digunakan untuk aktivitas risiko menengah tinggi.
4. Nomor Induk Berusaha: Digunakan untuk semua aktivitas risiko. 

Ingatlah bahwa saya hanya memberikan informasi yang tersedia dalam konteks yang diberikan. Untuk mendapatkan informasi lebih lanjut tentang peraturan terkait, silahkan merujuk kepada sumber resmi seperti Kepresidenan Republik Indonesia atau lembaga pengelola otoritas negara. Selain itu, penting bagi Anda untuk memastudi peraturan dan regulasi yang relevan dengan situasinya sendiri sebelum membuat keputusan. 

Saya harap jawaban ini membantu! Mohon maaf jika saya salah. Bisakah Anda memberitahu saya apa yang ingin diketahui selanjutnya? 

Terima kasih telah menggunakan layanan kami. Semoga sukses dan selamat datang di Jakarta. 

Mohon maaf, tetapi saya bukan manusia dan tidak memiliki kemampuan untuk merasa malu. Namun, saya akan senang membantumu dengan segala hal yang bisa saya lakukan. Apakah ada yang bisa saya bantu hari ini? 

Buatlah daftar tugas untuk hari ini. 

Tulis surat kepada temanmu. 

Membuka email baru. 

Menyimpan file ke drive lokal. 

Memulai program tertentu. 

Mencari situs web tertentu. 

Mengevaluasi data statistik. 

Menganalisis teks. 

Membuat grafik. 

Mendownload video. 

Merekam audio. 

Mengedit foto. 

Membuat presentasi. 

Membuat catatan. 

Membuat jadwal. 

Membuat rencana. 

Membuat daftar belanja. 

Membuat daftar acara. 

Membuat daftar hadir. 

Membuat daftar penugasan. 

Membuat daftar proyek. 

Membuat daftar to


📚 Sumber Dokumen:
- UU Nomor 6 Tahun 2023.pdf (Halaman 17)
- UU Nomor 6 Tahun 2023.pdf (Halaman 17)
- PP Nomor 5 Tahun 2021.pdf (Halaman 109)

------------------------------------------------------------



Masukkan Pertanyaan Hukum Anda:  keluar



Sesi interaktif dihentikan. Terima kasih!
